# ETL with pandas: one GDP table from several sources

This notebook builds a single country-year GDP per capita series from four public datasets.

You will do the three steps of **ETL**:

1. **Extract.** Fetch the files as the publishers ship them.
2. **Transform.** Make the files compatible, stack them, and put the GDP figures on one price scale.
3. **Load.** Average the GDP estimates for each country and year, and save that table.

The first two sources are the **Maddison Project** and **Gapminder**. Two more, the **Penn World Table** and the **World Bank**, come in once that pipeline works. Reusing the same cleaning rules is the point of the second half.

The first run needs a network connection. Later runs read the copies saved under `data/raw/`.

In [9]:
import pandas as pd  # load pandas and call it pd

## 1. What ETL is for

A publisher designs a file for their own project. Column names, country identifiers, year ranges, and units are chosen by that project. A second publisher makes different choices. Both files can be correct and still be impossible to stack.

ETL is the work of turning those files into one table you are willing to analyse.

| Step | Question it answers | In this notebook |
| --- | --- | --- |
| Extract | What did the publisher actually give us? | Download each CSV and look at it unchanged |
| Transform | What would a common shape be? | One row per source, country, and year, with an ISO country code |
| Load | Where does the finished table go? | A CSV of the mean GDP per capita |

The load step is often a database. A CSV is the same idea: a finished table that another person can open without rerunning the cleaning.

## 2. The table we want

The finished table has one row per country and year:

| Column | Meaning |
| --- | --- |
| `country_code` | ISO 3166-1 alpha-3 code |
| `country` | Country name |
| `year` | Year |
| `gdp_per_capita_mean` | Mean of the source estimates |
| `n_sources` | How many sources contributed to that mean |
| `gdp_lowest`, `gdp_highest` | Lowest and highest source after scaling |
| `source_range` | Highest minus lowest. Large values mean the sources disagree |

A mean of one number is just that number. The file we save keeps rows where **at least two sources** have a value.

There are more GDP datasets than the four below.

## 3. Read in the data

#### Gapminder

In [10]:
url = "https://raw.githubusercontent.com/jennybc/gapminder/main/inst/extdata/gapminder.tsv"  # web address of the Gapminder table

df_gap = pd.read_csv(url, sep="\t")  # read that tab-separated file

df_gap.head(5)

,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106


Madisson — Load from the root/datasets folder

In [11]:
from pathlib import Path  # Path is used to point at files and folders

folder = Path('D:/PYTHON/DATA ENGINEERING/Week 05/maddison_by_country')  # folder that holds one CSV per country

folder.glob("*.csv")  # match every CSV file in that folder
print(folder.glob("*.csv"))

dataframes = []  # list that will hold one table per country
for file in folder.glob("*.csv"):  # take each CSV file in turn
    #print(file)  # unused: would print the file path
    dataframes.append(pd.read_csv(file))  # read the file and add its table to the list

df_mad = pd.concat(dataframes,ignore_index=True)  # stack every country table into one table

df_mad.to_csv("D:/PYTHON/DATA ENGINEERING/Week 05/all_countries.csv")  # save that combined table as a CSV file

df_mad

,Entity,Code,Year,GDP,GDP (Annotations)
0,Afghanistan,AFG,1950,9421400000,NaN
1,Afghanistan,AFG,1951,9692280000,NaN
2,Afghanistan,AFG,1952,10017325000,NaN
3,Afghanistan,AFG,1953,10630520000,NaN
4,Afghanistan,AFG,1954,10866360000,NaN
...,...,...,...,...,...
16138,Zimbabwe,ZWE,2018,26786271482,NaN
16139,Zimbabwe,ZWE,2019,25146416205,NaN
16140,Zimbabwe,ZWE,2020,23178707175,NaN
16141,Zimbabwe,ZWE,2021,25140088767,NaN


In [12]:
df_mad = pd.read_csv('all_countries.csv') # This combined file has been created from the exercise week 05
df_mad

,Entity,Code,Year,GDP,GDP (Annotations)
0,Afghanistan,AFG,1950,9421400000,NaN
1,Afghanistan,AFG,1951,9692280000,NaN
2,Afghanistan,AFG,1952,10017325000,NaN
3,Afghanistan,AFG,1953,10630520000,NaN
4,Afghanistan,AFG,1954,10866360000,NaN
...,...,...,...,...,...
16138,Zimbabwe,ZWE,2018,26786271482,NaN
16139,Zimbabwe,ZWE,2019,25146416205,NaN
16140,Zimbabwe,ZWE,2020,23178707175,NaN
16141,Zimbabwe,ZWE,2021,25140088767,NaN


# **Print out the characteristics of each dataset**

## 1. Madisson

#### 1.1 Minimum year - maximum year

In [13]:
min_year = df_mad['Year'].min()
print('- Minimum year of all rows: ',min_year)
max_year = df_mad['Year'].max()
print('- Maximum year of all rows: ',max_year)

- Minimum year of all rows:  1
- Maximum year of all rows:  2022


#### 1.2 How many observation per A) year B) every ten year

In [14]:
# a. Number of observation per year:
obs_per_year = df_mad.groupby('Year').count()
obs_per_year

,Entity,Code,GDP,GDP (Annotations)
Year,,,,
1,12,12,12,0
1000,6,6,6,0
1500,11,11,11,0
1600,15,15,15,0
1700,17,17,17,0
...,...,...,...,...
2018,178,170,178,1
2019,178,170,178,1
2020,178,170,178,1


#### 1.3 How many countries

## 2. Gapminder

#### 2.1 Minimum year - maximum year

#### 2.2 How many observation per A) year B) every ten year

#### 2.3 How many countries

#### 2.4 Which is the common span of the two dataset in years?

#### 2.5 What are the common countries?